# Adding a Component to MegoBin — End-to-End Walkthrough

**Type:** Tutorial (public, shareable)  
**Audience:** A researcher who has just cloned MegoBin and wants to plug in a new component without touching the pipeline code.  
**Promoted from:** [`notebooks/concepts/MegoBin/04_python_protocols.ipynb`](../../concepts/MegoBin/04_python_protocols.ipynb), [`notebooks/concepts/MegoBin/02_hydra_composition_and_entry_point.ipynb`](../../concepts/MegoBin/02_hydra_composition_and_entry_point.ipynb), [`notebooks/concepts/MegoBin/05_end_to_end_experiment_pipeline.ipynb`](../../concepts/MegoBin/05_end_to_end_experiment_pipeline.ipynb).

## What you'll do

Create a new **binner** called `SemiBinBinner`, plug it into the pipeline beside the existing `UncertainGenRepresentation` and `CheckM2Evaluator`, and watch a full mock run go through every stage — all from this notebook, with simulated data, no HPC required.

We map each step to one pipeline stage so the Protocol/slot system stops feeling abstract:

```
          ┌──────── stages in megobin/pipeline.py ────────┐
Features → Representation → Binner → FASTA bins → Evaluator → results.tsv
  ↑              ↑            ↑                        ↑
  kmers+     UncertainGen   SemiBinBinner           CheckM2
  abundance  (existing)     (YOUR NEW COMPONENT)    (existing)
```

## Why this format

MegoBin's value prop is swappability: every stage is a Python `Protocol`, every component is a class with a `_target_` in a YAML, and the pipeline never needs to change. A new idea = **one new class + one new YAML + one edit to an experiment config**. Nothing else.

## Scope caveat

The HPC side (**DEIS-MCC**, **BioCloud**, **AI Lab**) isn't wired up yet in this notebook — we simulate data locally and stub the CheckM2 subprocess. Section 8 shows the one-line SBATCH invocation that will eventually run this same config on a real cluster.

## 1. Setup

Self-contained imports. Versions pinned to what's in the workspace `.venv`.

In [3]:
from __future__ import annotations

import sys
import textwrap
from pathlib import Path
from unittest.mock import patch

import numpy as np
import pandas as pd

from megobin.representations.base import Representation
from megobin.binners.base import Binner
from megobin.evaluators.base import Evaluator

print("python:   ", sys.executable)
print("numpy:    ", np.__version__)
print("pandas:   ", pd.__version__)


python:    /Users/tinnifreyr/Documents/Claude/Projects/Metagenomic Binning (Research Problem)/MegoBin/.venv/bin/python
numpy:     2.4.4
pandas:    3.0.2


## 2. Background — the Protocol/slot system in one picture

MegoBin doesn't use registries, plugins, or inheritance. A component is "in" if it **structurally** matches a `Protocol`. At runtime Hydra instantiates it from a YAML's `_target_` and hands it to the pipeline.

```
configs/binner/semibin.yaml             ← the YAML you write
   _target_: megobin.binners.semibin.SemiBinBinner
   k_neighbours: 200
            │
            │  hydra.utils.instantiate(cfg.binner)
            ▼
megobin/binners/semibin.py::SemiBinBinner    ← the class you write
   def cluster(embeddings) -> labels
            │
            │  structurally matches
            ▼
megobin/binners/base.py::Binner              ← the Protocol (unchanged)
```

The Protocols your component might implement (names + the single method that matters):

| Slot | Protocol | Your method | Shape in → Shape out |
|---|---|---|---|
| `representation` | `Representation` | `encode(features)` | `(N, input_dim) → (N, embedding_dim)` |
| `binner` | `Binner` | `cluster(embeddings)` | `(N, d) → (N,)` int labels |
| `evaluator` | `Evaluator` | `score(bins_dir)` | `Path → DataFrame(completeness, contamination)` |
| `loss` | `ContrastiveLoss` | `__call__(z_i, z_j, label)` | scalar tensor |
| `trainer` | `Trainer` | `fit(encoder, sampler, loss_fn)` | — |
| `logger` | `Logger` | `log_scalars`, `log_dataframe`, … | — |

That's the whole API surface. Today we're writing a `Binner`.

## 3. Stage 0 — Create a feature branch

MegoBin uses `feature/{component-name}`. Kebab-case, scoped to the single component you're adding.

| Adding… | Branch name |
|---|---|
| A binner called `SemiBinBinner` | `feature/semibin-binner` |
| A DNABERT-S encoder | `feature/dnabert-s-encoder` |
| A new contrastive loss | `feature/triplet-margin-loss` |

Shell commands you'd run in the MegoBin repo (don't run them from this notebook — shown for reference):

```bash
cd MegoBin
git checkout main && git pull
git checkout -b feature/semibin-binner
```

Rule of thumb: **one component per branch, one PR per branch.** Sweeps, ablations, and hypothesis tests live on separate `exp/H1-…` branches that compose existing components.

## 4. Stage 1 — Write the component class

Target file: [`MegoBin/megobin/binners/semibin.py`](../../../MegoBin/megobin/binners/) (the file doesn't exist yet — you'd create it on the branch).

Structure to match: the `Binner` Protocol at [`megobin/binners/base.py`](../../../MegoBin/megobin/binners/base.py). One method: `cluster(embeddings: np.ndarray) -> np.ndarray`.

Below is a minimal stub that's real enough to run — a SemiBin-style k-NN graph + connected-components binner. In a real PR you'd swap the body for SemiBin's actual Infomap+constraint routine, but the **Protocol shape is identical** either way, so the pipeline doesn't know or care.

In [4]:
# Contents of the file you'd commit at MegoBin/megobin/binners/semibin.py
# Inlined here so the notebook is self-contained.

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.neighbors import NearestNeighbors


class SemiBinBinner:
    """Connected components over a k-NN graph of embeddings.

    Matches the Binner Protocol — one method, one shape contract.
    """

    def __init__(self, k_neighbours: int = 20, min_bin_size: int = 2) -> None:
        self.k_neighbours = k_neighbours
        self.min_bin_size = min_bin_size

    def cluster(self, embeddings: np.ndarray) -> np.ndarray:
        n = embeddings.shape[0]
        k = min(self.k_neighbours, n - 1)
        knn = NearestNeighbors(n_neighbors=k + 1).fit(embeddings)
        _, idx = knn.kneighbors(embeddings)
        rows = np.repeat(np.arange(n), k)
        cols = idx[:, 1:].ravel()
        graph = csr_matrix((np.ones_like(rows), (rows, cols)), shape=(n, n))
        _, labels = connected_components(graph, directed=False)

        counts = np.bincount(labels)
        too_small = counts < self.min_bin_size
        labels = np.where(too_small[labels], -1, labels)
        return labels

## 5. Stage 2 — Verify Protocol conformance (before touching YAML)

Protocols are *structural*, so `isinstance(obj, Binner)` tells you at runtime whether your class is plug-compatible. This catches 90% of the "why doesn't the pipeline pick up my component?" errors at the keystroke they happen.

In [5]:
binner = SemiBinBinner(k_neighbours=5, min_bin_size=2)

assert isinstance(binner, Binner), "Shape mismatch — check cluster() signature"

toy_embeddings = np.vstack([
    np.random.randn(10, 8) + 0,
    np.random.randn(10, 8) + 6,
    np.random.randn(10, 8) + 12,
])
toy_labels = binner.cluster(toy_embeddings)

print("isinstance(binner, Binner):", isinstance(binner, Binner))
print("labels shape:              ", toy_labels.shape)
print("n bins:                    ", len(np.unique(toy_labels[toy_labels >= 0])))

isinstance(binner, Binner): True
labels shape:               (30,)
n bins:                     3


Three well-separated clusters → three bins. The component passes the protocol gate.

**If the `isinstance` assertion fires,** nine times out of ten one of:
- the method is named `fit` or `predict` instead of `cluster`,
- the return type is a list instead of `np.ndarray`,
- the signature has the wrong number of positional args.

## 6. Stage 3 — Write the component YAML

Target file: `MegoBin/configs/binner/semibin.yaml`. This is the file Hydra resolves when you write `binner: semibin` in a defaults list.

The whole file is four lines — `_target_` plus your constructor kwargs:

In [6]:
binner_yaml = textwrap.dedent("""
    _target_: megobin.binners.semibin.SemiBinBinner
    k_neighbours: 20
    min_bin_size: 2
""").strip()

print("# MegoBin/configs/binner/semibin.yaml")
print(binner_yaml)

# MegoBin/configs/binner/semibin.yaml
_target_: megobin.binners.semibin.SemiBinBinner
k_neighbours: 20
min_bin_size: 2


Two rules that trip people up:

1. **`_target_` is a Python import path, not a file path** — dots, not slashes, and it must point to the class (not the module).
2. **Every kwarg must be present** with a default — no `_partial_` magic unless you intend partial instantiation (see `two_phase.yaml` for that pattern).

## 7. Stage 4 — Compose it into an experiment config

Target file: `MegoBin/configs/experiment/uncertain_gen_semibin_checkm2.yaml`. This is the config you'd pass to `megobin/pipeline.py` with `--config-name`.

The `defaults` list composes slots; `_self_` lets you override specific fields below.

In [7]:
experiment_yaml = textwrap.dedent("""
    # @package _global_
    defaults:
      - _self_
      - /dataset: CAMI_toy
      - /features: canonical_kmer_abundance
      - /representation: uncertain_gen     # existing
      - /loss: mahalanobis_bce             # existing
      - /binner: semibin                   # YOUR NEW COMPONENT
      - /evaluator: checkm2                # existing
      - /pair_sampler: hybrid
      - /trainer: two_phase
      - /logger: tensorboard

    seed: 42
    use_abundance: true

    representation:
      input_dim: 236
      embedding_dim: 256
""").strip()

print("# MegoBin/configs/experiment/uncertain_gen_semibin_checkm2.yaml")
print(experiment_yaml)

# MegoBin/configs/experiment/uncertain_gen_semibin_checkm2.yaml
# @package _global_
defaults:
  - _self_
  - /dataset: CAMI_toy
  - /features: canonical_kmer_abundance
  - /representation: uncertain_gen     # existing
  - /loss: mahalanobis_bce             # existing
  - /binner: semibin                   # YOUR NEW COMPONENT
  - /evaluator: checkm2                # existing
  - /pair_sampler: hybrid
  - /trainer: two_phase
  - /logger: tensorboard

seed: 42
use_abundance: true

representation:
  input_dim: 236
  embedding_dim: 256


You can also skip the experiment file and override from the CLI:

```bash
python megobin/pipeline.py binner=semibin representation=uncertain_gen evaluator=checkm2
```

Same result. Useful for one-off sanity checks before committing a config.

## 8. Stage 5 — Simulate the data the pipeline would load

The real pipeline reads `kmer_profiles.npy` and `abundance.npy` from `dataset.path`. We'll fake three genomes' worth of contigs with **distinct** k-mer + abundance signatures so the representation has something learnable and the binner has something clusterable.

Shapes match [`configs/features/canonical_kmer_abundance.yaml`](../../../MegoBin/configs/features/canonical_kmer_abundance.yaml): 136 canonical 4-mer dims + 100 abundance dims = 236 input dims.

In [8]:
rng = np.random.default_rng(42)
N_GENOMES = 3
CONTIGS_PER_GENOME = 40
KMER_DIM = 136
ABUND_DIM = 100

features_list, truth_list = [], []
for g in range(N_GENOMES):
    kmer_centre = rng.dirichlet(np.ones(KMER_DIM) * 0.5)
    kmers = kmer_centre + 0.01 * rng.standard_normal((CONTIGS_PER_GENOME, KMER_DIM))
    kmers = np.clip(kmers, 1e-6, None)
    kmers /= kmers.sum(axis=1, keepdims=True)

    abund_centre = rng.uniform(1, 10, size=ABUND_DIM)
    abund = abund_centre + 0.3 * rng.standard_normal((CONTIGS_PER_GENOME, ABUND_DIM))

    features_list.append(np.concatenate([kmers, abund], axis=1))
    truth_list.append(np.full(CONTIGS_PER_GENOME, g))

features = np.concatenate(features_list, axis=0).astype(np.float32)
truth = np.concatenate(truth_list)

print("features.shape:", features.shape, "(n_contigs, 236)")
print("truth.shape:   ", truth.shape, "→", np.bincount(truth).tolist())

features.shape: (120, 236) (n_contigs, 236)
truth.shape:    (120,) → [40, 40, 40]


## 9. Stage 6 — Run the pipeline stages in order (mocked)

Rather than launching `megobin/pipeline.py` (which wants DVC-tracked data + a real CheckM2 binary), we walk through exactly what [`megobin/pipeline.py`](../../../MegoBin/megobin/pipeline.py) does, stage by stage, with print statements so you can see each handoff.

### 9a. Representation — encode features to embeddings

`pipeline.py:108` instantiates `cfg.representation`; `pipeline.py:179` calls `.encode(features)`. We skip training here (mock run) and just init the encoder untrained — enough to show the plumbing.

In [9]:
from megobin.representations.uncertain_gen import UncertainGenRepresentation

representation = UncertainGenRepresentation(
    input_dim=236, hidden_dim=512, embedding_dim=256, dropout=0.2,
)
assert isinstance(representation, Representation)

embeddings = representation.encode(features)
print("embeddings.shape:", embeddings.shape, "(n_contigs, 256)")

embeddings.shape: (120, 256) (n_contigs, 256)


### 9b. Binner — cluster embeddings into bin labels

`pipeline.py:186` calls `binner.cluster(embeddings)`. This is the single call your `SemiBinBinner` is responsible for.

Because the untrained encoder produces near-random embeddings, we cluster the raw features instead to illustrate "the binner works when the representation is informative." In a real run the trained encoder produces separable embeddings and the binner runs on those.

In [10]:
labels = binner.cluster(features)

print("labels:", labels[:20], "…")
print("n bins (excl. noise):", len(np.unique(labels[labels >= 0])))
print("n noise contigs:     ", int((labels == -1).sum()))

labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0] …
n bins (excl. noise): 3
n noise contigs:      0


### 9c. Write bins as FASTA files

`pipeline.py:188-198` materialises each bin to disk as one FASTA file — that's what CheckM2 expects as input. In a real run the pipeline gets contig sequences from the dataset; we stub short dummy sequences here.

In [11]:
bins_dir = Path("/tmp/megobin_mock_run/bins")
bins_dir.mkdir(parents=True, exist_ok=True)
for old in bins_dir.glob("*.fa"):
    old.unlink()

for bin_id in np.unique(labels[labels >= 0]):
    members = np.where(labels == bin_id)[0]
    with (bins_dir / f"bin_{bin_id}.fa").open("w") as f:
        for i in members:
            f.write(f">contig_{i}\nACGT\n")

fasta_files = sorted(bins_dir.glob('*.fa'))
print("bins_dir:", bins_dir)
print("files:   ", [p.name for p in fasta_files])

bins_dir: /tmp/megobin_mock_run/bins
files:    ['bin_0.fa', 'bin_1.fa', 'bin_2.fa']


### 9d. Evaluator — CheckM2 on the bin directory (mocked subprocess)

`pipeline.py` calls `evaluator.score(bins_dir)`, which under the hood shells out to `checkm2 predict`. CheckM2 isn't installed in this notebook's venv (it's a BioCloud-side tool), so we mock the subprocess the same way [`MegoBin/tests/test_interfaces.py`](../../../MegoBin/tests/test_interfaces.py) does — intercept `subprocess.run`, drop a fake `quality_report.tsv`, let the evaluator's parser handle the rest.

This is exactly how you'd add unit tests for a new evaluator.

In [13]:
from megobin.evaluators.checkm2 import CheckM2Evaluator

evaluator = CheckM2Evaluator(threads=1)
assert isinstance(evaluator, Evaluator)


def fake_checkm2(cmd, *args, **kwargs):
    output_dir = Path(cmd[cmd.index("-o") + 1])
    output_dir.mkdir(parents=True, exist_ok=True)
    rows = [
        f"{p.stem}\t{rng.uniform(70, 99):.1f}\t{rng.uniform(0, 5):.1f}"
        for p in fasta_files
    ]
    (output_dir / "quality_report.tsv").write_text(
        "Name\tCompleteness\tContamination\n" + "\n".join(rows)
    )
    class R:
        returncode = 0
        stdout = ""
        stderr = ""
    return R()

with patch("megobin.evaluators.checkm2.subprocess.run", side_effect=fake_checkm2):
    results = evaluator.score(bins_dir)

results

,completeness,contamination
name,,
bin_0,89.0,1.0
bin_1,96.1,3.4
bin_2,79.7,2.0


That DataFrame is the pipeline's final artifact — written to `outputs/<run>/results.tsv` and logged to TensorBoard via `logger.log_dataframe("results", df)`.

## 10. Stage 7 — What a real run looks like

On a workstation with the `.venv` active and CheckM2 installed, the entire flow above collapses to one command. Hydra picks up your YAML, instantiates every slot, runs every stage.

```bash
cd MegoBin
python megobin/pipeline.py --config-name experiment/uncertain_gen_semibin_checkm2
```

On HPC, the same command is wrapped in an SBATCH script under [`MegoBin/hpc/slurm/`](../../../MegoBin/hpc/slurm/). The cluster-specific bits (container flags, partitions) are **not wired up yet** in this workspace — when they are, the flow per cluster is:

| Cluster | Role | Container flag (documented) |
|---|---|---|
| **AI Lab (AAU)** | Primary compute | native `.venv` |
| **BioCloud (CMC-AAU)** | Feature computation + CheckM2 eval | `apptainer run --nvccli` |
| **DEIS-MCC** | GPU training (T4 / L4) | `singularity exec --nv` |

Outputs (`tb/`, `checkpoints/`, `results.tsv`) get `rsync`'d back; TensorBoard points at the local tree to compare runs.

## 11. Stage 8 — Commit, push, open a PR

```bash
cd MegoBin
git add megobin/binners/semibin.py configs/binner/semibin.yaml configs/experiment/uncertain_gen_semibin_checkm2.yaml
git add tests/test_interfaces.py   # add a Protocol conformance test for SemiBinBinner
git commit -m "feat(binner): add SemiBinBinner + config"
git push -u origin feature/semibin-binner
gh pr create --fill
```

Checklist before the PR is reviewable:

- [ ] `isinstance(component, Protocol)` passes in a test.
- [ ] Overfit test: `pytest tests/test_overfit_batch.py` — smoke test a representation; adapt pattern for binner by checking ARI ≥ 0.9 on synthetic separable embeddings.
- [ ] Experiment config runs end-to-end on `CAMI_toy`.
- [ ] TensorBoard event file exists in the run's `tb/`.

## 12. Experiments to try (fork-off cells)

Each of these is one-line swap via Hydra CLI overrides — no code edits:

```bash
# Swap binner, keep everything else
python megobin/pipeline.py --config-name experiment/uncertain_gen_semibin_checkm2 binner=infomap

# Swap representation to SemiBin's encoder; see if UncertainGen was actually helping
python megobin/pipeline.py --config-name experiment/uncertain_gen_semibin_checkm2 representation=semibin_encoder

# Sweep k_neighbours
python megobin/pipeline.py --config-name experiment/uncertain_gen_semibin_checkm2 \
    binner.k_neighbours=10,20,50,100 --multirun

# Silence the logger for fast iteration
python megobin/pipeline.py --config-name experiment/uncertain_gen_semibin_checkm2 logger=none
```

A hypothesis test (`Hn`) gets its own `exp/H<n>-<slug>` branch that composes existing components via a new experiment YAML — don't put hypothesis-specific logic inside a reusable component.

## 13. Further reading

- **The Protocols:** [`MegoBin/megobin/representations/base.py`](../../../MegoBin/megobin/representations/base.py), [`MegoBin/megobin/binners/base.py`](../../../MegoBin/megobin/binners/base.py), [`MegoBin/megobin/evaluators/base.py`](../../../MegoBin/megobin/evaluators/base.py), [`MegoBin/megobin/losses/base.py`](../../../MegoBin/megobin/losses/base.py), [`MegoBin/megobin/utils/logger.py`](../../../MegoBin/megobin/utils/logger.py).
- **A real Binner with non-trivial logic:** [`MegoBin/megobin/binners/infomap.py`](../../../MegoBin/megobin/binners/infomap.py).
- **A real subprocess-wrapping Evaluator:** [`MegoBin/megobin/evaluators/checkm2.py`](../../../MegoBin/megobin/evaluators/checkm2.py).
- **Pipeline entry point:** [`MegoBin/megobin/pipeline.py`](../../../MegoBin/megobin/pipeline.py).
- **Concept notebook this was promoted from:** [`notebooks/concepts/MegoBin/05_end_to_end_experiment_pipeline.ipynb`](../../concepts/MegoBin/05_end_to_end_experiment_pipeline.ipynb).

Obsidian: see `Research/01-Literature/Concepts/Python Protocols` and `Research/01-Literature/Concepts/Hydra Composition` for the theory side of the Protocol/slot architecture.

## 14. Conclusion

Adding a component to MegoBin is **one class + one YAML + one defaults-list edit.** You never touch `pipeline.py`. The Protocol gives you a compile-time-ish contract that `isinstance` enforces at runtime; Hydra gives you the wiring; the researcher gives the idea.

The same stages you walked through here will run unchanged once the HPC pipes are connected — the only thing that changes is where the process executes.